# Dataset overview

Inspect the column names, data types, and full row counts of the two local Parquet shards. The combined count is the sum of both shards.

For an explicitly owner-authorized pre-seal inspection, select the repository's `.venv` Python kernel and run the cells in order. The notebook supports a working directory of either the repository root or `notebooks/`.

**Historical pre-seal workflow:** issue #22 inspected the source before the
[Test-first policy](../docs/dataset.md#preparation-and-splitting-rules) was adopted.
After Test membership is sealed, do not run this whole-source notebook: full-source
previews, label summaries, and image samples can expose Test records. Use a future
Development-only loader/workflow instead. Issue #30 changes explanatory Markdown
only; no runtime seal check or access guard is implemented here. The existing
code still reads the full raw shards. Real-data execution requires separate owner
authorization under [AGENTS.md](../AGENTS.md).


## Locate the local files

Resolve paths relative to the repository. Missing files produce an error before inspection begins.


In [ ]:
from pathlib import Path

import pyarrow.parquet as pq

working_directory = Path.cwd().resolve()
ROOT = (
    working_directory.parent
    if working_directory.name == "notebooks"
    else working_directory
)

if not (
    (ROOT / "pyproject.toml").is_file()
    and (ROOT / "src" / "vlm_finetuning").is_dir()
):
    raise FileNotFoundError(
        "Start the notebook from the repository root or its notebooks/ directory."
    )

DATA_DIR = ROOT / "data" / "raw"
files = [
    DATA_DIR / "train-00000-of-00002-6cff4c59f91661c3.parquet",
    DATA_DIR / "train-00001-of-00002-bb459e5ac5f01e71.parquet",
]

for path in files:
    if not path.resolve().is_relative_to(ROOT):
        raise ValueError(f"Dataset file resolves outside the repository: {path.name}")
    if not path.is_file():
        raise FileNotFoundError(f"Missing local Parquet shard: {path}")

print(f"Dataset directory: {DATA_DIR.relative_to(ROOT)}")
print(f"Number of shards: {len(files)}")


## Inspect each shard

Display every column name and data type from the Parquet schema. Count all rows by reading only the `id` column in batches, then compare that count with the Parquet metadata. Image payloads are not decoded.


In [ ]:
summary = []

for path in files:
    with pq.ParquetFile(path) as parquet:
        schema = parquet.schema_arrow
        metadata_rows = parquet.metadata.num_rows

        print(f"\nFile: {path.name}")
        print(f"Column count: {len(schema)}")
        print("Columns:")
        for number, field in enumerate(schema, start=1):
            print(f"  {number}. {field.name}: {field.type}")

        if "id" not in schema.names:
            raise ValueError(f"Required counting column 'id' is missing: {path.name}")

        scanned_rows = sum(
            batch.num_rows
            for batch in parquet.iter_batches(
                columns=["id"],
                batch_size=4096,
                use_threads=False,
            )
        )

        if scanned_rows != metadata_rows:
            raise ValueError(
                f"Row-count mismatch in {path.name}: "
                f"scanned={scanned_rows}, metadata={metadata_rows}"
            )

        print(f"Full scanned row count: {scanned_rows:,}")
        print(f"Parquet metadata row count: {metadata_rows:,}")

        summary.append(
            {
                "file": path.name,
                "columns": len(schema),
                "rows": scanned_rows,
            }
        )


## Combined row count

Summarize each shard and add the scanned row counts to obtain the full dataset row count.


In [ ]:
print(f"{'File':<55} {'Columns':>8} {'Rows':>12}")

for shard in summary:
    print(f"{shard['file']:<55} {shard['columns']:>8} {shard['rows']:>12,}")

total_rows = sum(shard["rows"] for shard in summary)
print(f"\nCombined dataset row count: {total_rows:,}")


## First 100 raw rows from each shard

Display a separate table for each Parquet shard, including every column name and the first 100 rows in source order. All 100 rows and all columns are displayed. Long cell values, such as the image bytes, have shortened text representations; the DataFrames retain their raw values. Images are not decoded.


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

raw_previews = {}

for path in files:
    with pq.ParquetFile(path) as parquet:
        first_batch = next(
            parquet.iter_batches(batch_size=100, use_threads=False),
            None,
        )
        preview = (
            first_batch.to_pandas()
            if first_batch is not None
            else pd.DataFrame(columns=parquet.schema_arrow.names)
        )

    raw_previews[path.name] = preview
    display(Markdown(f"### {path.name}\n\nShowing {len(preview):,} raw rows."))

    with pd.option_context(
        "display.max_rows", 100,
        "display.max_columns", None,
        "display.max_colwidth", 80,
    ):
        display(preview)


## Compare IDs across both shards

Read the complete `id` column from each file, including rows beyond the 100-row previews. Show missing IDs and repeated IDs within each shard, then compare the sets of non-null IDs across both files. Shared and file-only counts refer to distinct IDs. This checks product IDs, not image similarity or equality of other column values.


In [ ]:
from collections import Counter

import pandas as pd
from IPython.display import display

if len(files) != 2:
    raise ValueError("This comparison requires exactly two Parquet shards.")

id_counts_by_file = {}
id_summary_rows = []

for path in files:
    counts = Counter()
    rows_checked = 0
    missing_ids = 0

    with pq.ParquetFile(path) as parquet:
        if "id" not in parquet.schema_arrow.names:
            raise ValueError(f"Required column 'id' is missing: {path.name}")

        for batch in parquet.iter_batches(
            columns=["id"],
            batch_size=4096,
            use_threads=False,
        ):
            rows_checked += batch.num_rows
            for value in batch.column(0).to_pylist():
                if value is None:
                    missing_ids += 1
                else:
                    counts[value] += 1

    id_counts_by_file[path.name] = counts
    id_summary_rows.append(
        {
            "File": path.name,
            "Rows checked": rows_checked,
            "Missing IDs": missing_ids,
            "Unique non-null IDs": len(counts),
            "Repeated IDs": sum(count > 1 for count in counts.values()),
            "Extra rows sharing IDs": sum(count - 1 for count in counts.values()),
        }
    )

id_summary = pd.DataFrame(id_summary_rows)
with pd.option_context("display.max_columns", None):
    display(id_summary)

first_ids = set(id_counts_by_file[files[0].name])
second_ids = set(id_counts_by_file[files[1].name])
shared_ids = first_ids & second_ids
only_first_ids = first_ids - second_ids
only_second_ids = second_ids - first_ids

id_comparison = pd.DataFrame(
    [
        {"Comparison": "Shared IDs across both files", "Distinct ID count": len(shared_ids)},
        {"Comparison": "IDs only in the first file", "Distinct ID count": len(only_first_ids)},
        {"Comparison": "IDs only in the second file", "Distinct ID count": len(only_second_ids)},
    ]
)
display(id_comparison)

if not first_ids and not second_ids:
    print("Neither file contains a non-null ID.")
elif first_ids == second_ids:
    print("Both files contain the same set of non-null IDs.")
elif shared_ids:
    print("The files share some non-null IDs; their ID sets differ.")
else:
    print("No shared non-null IDs were found between the files.")

if shared_ids:
    display(pd.DataFrame({"Shared ID (first 20)": sorted(shared_ids)[:20]}))


## Random 10 images and raw metadata from each shard

Select 10 distinct random row positions from each complete shard, then display each image with a one-row table containing all original columns and raw values. The selected rows are displayed in source order. Set `IMAGE_SAMPLE_SEED` to a different integer for another repeatable sample.

Parquet batches are read until the selected rows are collected; only the selected images are decoded. Long values in the metadata tables, including image bytes, have shortened text representations. The complete selected records remain available in `random_image_samples`. Source files are unchanged.


In [ ]:
import random
from io import BytesIO

import pandas as pd
from IPython.display import Markdown, display
from PIL import Image

IMAGES_PER_SHARD = 10
IMAGE_SAMPLE_SEED = 42
random_image_samples = {}

for shard_index, path in enumerate(files):
    samples = []

    with pq.ParquetFile(path) as parquet:
        column_names = parquet.schema_arrow.names
        if "image" not in column_names:
            raise ValueError(f"Required column 'image' is missing: {path.name}")

        row_count = parquet.metadata.num_rows
        rng = random.Random(IMAGE_SAMPLE_SEED + shard_index)
        sampled_positions = sorted(
            rng.sample(range(row_count), min(IMAGES_PER_SHARD, row_count))
        )

        if sampled_positions:
            row_offset = 0
            for batch in parquet.iter_batches(batch_size=512, use_threads=False):
                batch_end = row_offset + batch.num_rows
                for row_position in sampled_positions:
                    if row_offset <= row_position < batch_end:
                        record = batch.slice(row_position - row_offset, 1).to_pylist()[0]
                        samples.append((row_position, record))
                row_offset = batch_end
                if len(samples) == len(sampled_positions):
                    break

        if len(samples) != len(sampled_positions):
            raise ValueError(f"Could not read every selected row from {path.name}")

    random_image_samples[path.name] = samples
    display(Markdown(f"### {path.name}\n\nSelected images: {len(samples)}"))

    for sample_number, (row_position, record) in enumerate(samples, start=1):
        display(
            Markdown(
                f"#### Image {sample_number} — source row {row_position}, "
                f"ID {record.get('id')}"
            )
        )

        payload = (record["image"] or {}).get("bytes")
        if not payload:
            raise ValueError(
                f"Missing embedded image bytes: {path.name}, source row {row_position}"
            )

        with Image.open(BytesIO(payload)) as image:
            image.load()
            display(image.copy())

        metadata_table = pd.DataFrame(
            [record],
            columns=column_names,
            index=pd.Index([row_position], name="source_row"),
        )
        with pd.option_context(
            "display.max_columns", None,
            "display.max_colwidth", 80,
        ):
            display(metadata_table)
